# Fatia 02 — Landing ➜ Bronze (Faturamento, Rede & Atendimento)

**Dono:** Ryei Estevan

Ingere as 4 fontes da Fatia 02 **sem alterar conteúdo** (regra Bronze):

| Fonte | Formato | Tabela Delta |
|---|---|---|
| `faturas.csv` | CSV `;` (mainframe) | `bronze_faturas` |
| `ocorrencias_rede.csv` | CSV `,` | `bronze_ocorrencias_rede` |
| `atendimentos.json` | JSON array multilinha | `bronze_atendimentos` |
| `inspecoes.xlsx` | Excel, 3 abas | `bronze_inspecoes` |
| `aneel_limites_continuidade.csv` | CSV `;` (ANEEL) | `bronze_aneel_limites` |

Todas as colunas de origem são lidas como texto (exceto a estrutura aninhada do JSON),
e cada tabela recebe `_ingested_at` e `_source_file`.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
import os
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_DATABRICKS:
    CATALOG = "visagio_atividade_v_energy_360"
    SCHEMA = "bronze"
    BRONZE = f"{CATALOG}.{SCHEMA}"
    LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/landing"
else:
    from local_runtime import setup_local_env
    spark, dbutils, display, _paths, _schemas = setup_local_env()
    CATALOG = _schemas["catalog"]
    SCHEMA = _schemas["bronze"]
    BRONZE = _schemas["bronze"]
    LANDING_PATH = _paths["landing"]

In [0]:

def add_audit_columns(raw_df: DataFrame) -> DataFrame:
    """Anexa metadados técnicos obrigatórios da camada Bronze."""
    return raw_df.withColumn("_ingested_at", F.current_timestamp()).withColumn(
        "_source_file", F.col("_metadata.file_path")
    )


def save_bronze(bronze_df: DataFrame, table_name: str) -> None:
    full_name = f"{BRONZE}.{table_name}"
    (
        bronze_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    print(f"✓ {full_name}: {spark.table(full_name).count():,} linhas")


def read_delimited(file_name: str, separator: str) -> DataFrame:
    return (
        spark.read.option("header", "true")
        .option("inferSchema", "false")
        .option("sep", separator)
        .csv(f"{LANDING_PATH}/{file_name}")
    )

## Faturas e Ocorrências de Rede (CSV)

In [0]:
save_bronze(add_audit_columns(read_delimited("faturas.csv", ";")), "bronze_faturas")
save_bronze(
    add_audit_columns(read_delimited("ocorrencias_rede.csv", ",")),
    "bronze_ocorrencias_rede",
)

## Atendimentos (JSON aninhado — `historico` preservado como Array<Struct>)

In [0]:
atendimentos_df = spark.read.option("multiLine", "true").json(
    f"{LANDING_PATH}/atendimentos.json"
)
save_bronze(add_audit_columns(atendimentos_df), "bronze_atendimentos")

## Inspeções (Excel — 3 abas empilhadas, aba de origem em `_source_sheet`)

Spark não lê `.xlsx` nativamente no Serverless, então cada aba é lida com pandas
como texto puro (`dtype=str`) para não converter datas nem decimais com vírgula.

In [0]:
import pandas as pd

inspecoes_path = f"{LANDING_PATH}/inspecoes.xlsx"
sheets_by_name = pd.read_excel(inspecoes_path, sheet_name=None, dtype=str)
inspecoes_pdf = pd.concat(
    [sheet_pdf.assign(_source_sheet=sheet_name) for sheet_name, sheet_pdf in sheets_by_name.items()],
    ignore_index=True,
)
inspecoes_df = (
    spark.createDataFrame(inspecoes_pdf.astype(object).where(inspecoes_pdf.notna(), None))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(inspecoes_path))
)
save_bronze(inspecoes_df, "bronze_inspecoes")

## Limites ANEEL de DEC/FEC (fonte externa — ADR-003)

`indicadores-continuidade-coletivos-limite.csv` do Portal de Dados Abertos da ANEEL,
com os limites anuais por conjunto elétrico. Versionado no repositório em
`data-engineering/reference/aneel_limites_continuidade.csv`, só com as linhas da Neoenergia PE
(o arquivo não vem nos zips do case); deve ser enviado ao Volume `landing` com os demais.

In [0]:
save_bronze(
    add_audit_columns(read_delimited("aneel_limites_continuidade.csv", ";")),
    "bronze_aneel_limites",
)

## Verificação rápida

In [0]:
BRONZE_TABLES = [
    "bronze_faturas",
    "bronze_ocorrencias_rede",
    "bronze_atendimentos",
    "bronze_inspecoes",
    "bronze_aneel_limites",
]
for table_name in BRONZE_TABLES:
    spark.table(f"{BRONZE}.{table_name}").printSchema()